In [1]:
import os
import json

class KafkaPartitionDemo:
    def __init__(self, partition_file="partition_0.log"):
        """Инициализация партиции (Append-only файла на диске)"""
        self.filepath = partition_file
        if not os.path.exists(self.filepath):
            open(self.filepath, 'w').close()

    def append_message(self, message: dict):
        """
        ПРОДЮСЕР (Producer): Дописывает сообщение в конец файла.
        Возвращает оффсет (номер байта, с которого начинается сообщение).
        """
        payload = json.dumps(message) + "\n"
        
        with open(self.filepath, 'a', encoding='utf-8') as f:
            offset = f.tell() 
            f.write(payload)
            f.flush()
            os.fsync(f.fileno())          
        return offset

    def get_messages_from_offset(self, offset: int, max_messages=5):
        """
        КОНСЬЮМЕР (Consumer): Читает пачку сообщений, начиная с конкретного байта.
        """
        messages = []
        with open(self.filepath, 'r', encoding='utf-8') as f:
            f.seek(offset)
            
            for _ in range(max_messages):
                line = f.readline()
                if not line:
                    break
                messages.append(json.loads(line))
            
            new_offset = f.tell()           
        return messages, new_offset


if __name__ == "__main__":
    partition = KafkaPartitionDemo()
    
    print("\n[PRODUCER] Заваливаем лог транзакциями...")
    partition.append_message({"tx_id": 1, "user": "Alex", "amount": 100})
    partition.append_message({"tx_id": 2, "user": "Bob", "amount": 250})
    partition.append_message({"tx_id": 3, "user": "Charlie", "amount": 50})
    
    # ----------------------------------------------------
    print("\n--- Сценарий: Два независимых микросервиса ---")
    
    print("\n1. [БИЛЛИНГ] Читает с самого начала (Offset 0):")
    billing_offset = 0
    msgs, billing_offset = partition.get_messages_from_offset(billing_offset, max_messages=2)
    for m in msgs: print(f"Биллинг обработал: {m}")
    print(f"Биллинг запомнил свой новый offset: {billing_offset}")
    
    print("\n2. [СИСТЕМА УВЕДОМЛЕНИЙ] Тоже начинает с начала (Offset 0):")
    push_offset = 0
    # Читает сразу все 3 сообщения (Kafka позволяет читать батчами)
    msgs, push_offset = partition.get_messages_from_offset(push_offset, max_messages=3)
    for m in msgs: print(f"Push-сервис отправил СМС: {m}")
    print(f"Push-сервис ушел вперед. Его offset: {push_offset}")
    
    # ----------------------------------------------------
    print("\n[PRODUCER] Прилетает новая транзакция в онлайне...")
    partition.append_message({"tx_id": 4, "user": "Dave", "amount": 900})
    
    print("\n3. [БИЛЛИНГ] Просыпается и продолжает работу со своего старого offset-а:")
    # Биллинг не читает то, что уже обработал!
    msgs, billing_offset = partition.get_messages_from_offset(billing_offset, max_messages=2)
    for m in msgs: print(f"Биллинг дообработал: {m}")
    
    # Очистка за собой
    os.remove("partition_0.log")



[PRODUCER] Заваливаем лог транзакциями...

--- Сценарий: Два независимых микросервиса ---

1. [БИЛЛИНГ] Читает с самого начала (Offset 0):
Биллинг обработал: {'tx_id': 1, 'user': 'Alex', 'amount': 100}
Биллинг обработал: {'tx_id': 2, 'user': 'Bob', 'amount': 250}
Биллинг запомнил свой новый offset: 87

2. [СИСТЕМА УВЕДОМЛЕНИЙ] Тоже начинает с начала (Offset 0):
Push-сервис отправил СМС: {'tx_id': 1, 'user': 'Alex', 'amount': 100}
Push-сервис отправил СМС: {'tx_id': 2, 'user': 'Bob', 'amount': 250}
Push-сервис отправил СМС: {'tx_id': 3, 'user': 'Charlie', 'amount': 50}
Push-сервис ушел вперед. Его offset: 133

[PRODUCER] Прилетает новая транзакция в онлайне...

3. [БИЛЛИНГ] Просыпается и продолжает работу со своего старого offset-а:
Биллинг дообработал: {'tx_id': 3, 'user': 'Charlie', 'amount': 50}
Биллинг дообработал: {'tx_id': 4, 'user': 'Dave', 'amount': 900}


In [3]:
import os

FILE_NAME = "kafka_topic.log"

def create_kafka_stream():
    """Эмулятор шлюза: закидываем в Кафку 100 000 транзакций"""
    print("1. [PRODUCER] Клиенты совершают покупки. Пишем в Append-Only лог...")
    with open(FILE_NAME, "w") as f:
        for i in range(1, 100_001):
            f.write(f"Транзакция_{i}: 100 руб.\n")
    print("Лог записан на жесткий диск. Файл закрыт.\n")

def read_realtime(offset):
    """Эмулятор микросервиса, который читает свежие новости"""
    print(f"2. [СЕРВИС АНАЛИТИКИ] Читает новые события с конца (Offset: {offset})")
    with open(FILE_NAME, "r") as f:
        f.seek(offset)
        # Читаем только 3 последних строчки
        for _ in range(3):
            print(" ->", f.readline().strip())

def rewind_time():
    """Эмулятор того самого 'Путешествия во времени'"""
    print("\n3. алярма")
    print("База данных упала 5 часов назад. Мы потеряли часть транзакций!")
    print("Если бы это был RabbitMQ - деньги клиентов исчезли бы навсегда.")
    print("Но мы в Kafka. Запускаем скрипт восстановления...\n")
    
    print("4. [СКРИПТ ВОССТАНОВЛЕНИЯ] Устанавливаем Offset = 0 (Начало времен)")
    with open(FILE_NAME, "r") as f:
        f.seek(0)
        for _ in range(3):
            print(" -> [ВОССТАНОВЛЕНО]:", f.readline().strip())
    print("... и так далее, пока не восстановим все 100 000 записей!")

if __name__ == "__main__":
    create_kafka_stream()
    
    file_size = os.path.getsize(FILE_NAME)

    read_realtime(offset=file_size - 100)
    
    rewind_time()
    
    os.remove(FILE_NAME)


1. [PRODUCER] Клиенты совершают покупки. Пишем в Append-Only лог...
Лог записан на жесткий диск. Файл закрыт.

2. [СЕРВИС АНАЛИТИКИ] Читает новые события с конца (Offset: 3988795)
 -> 99998: 100 руб.
 -> Транзакция_99999: 100 руб.
 -> Транзакция_100000: 100 руб.

3. алярма
База данных упала 5 часов назад. Мы потеряли часть транзакций!
Если бы это был RabbitMQ - деньги клиентов исчезли бы навсегда.
Но мы в Kafka. Запускаем скрипт восстановления...

4. [СКРИПТ ВОССТАНОВЛЕНИЯ] Устанавливаем Offset = 0 (Начало времен)
 -> [ВОССТАНОВЛЕНО]: Транзакция_1: 100 руб.
 -> [ВОССТАНОВЛЕНО]: Транзакция_2: 100 руб.
 -> [ВОССТАНОВЛЕНО]: Транзакция_3: 100 руб.
... и так далее, пока не восстановим все 100 000 записей!
